In [4]:
from __future__ import print_function
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import torchvision.models as models
import os, sys, json
import cv2
from PIL import Image
import datetime

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add project root directory to sys.path
# !!! IMPORTANT: Replace 'YOUR_PROJECT_FOLDER_NAME' with the actual name of your project folder on Google Drive.
# For example, if your project is at /content/drive/MyDrive/MyAwesomeProject, then set project_folder_name = 'MyAwesomeProject'
project_folder_name = 'ball_tracking_colabo' # <--- Please update this with your actual folder name!
project_path = os.path.join('/content/drive/MyDrive', project_folder_name)
sys.path.append(project_path)

from models.unet import UNet

from utils.detector import detect
import glob

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:

cuda0 = torch.device('cuda:0')
model = UNet(27).to(cuda0)

model.load_state_dict(torch.load(os.path.join(project_path, 'weight/20211107/epoch_27_4030')))
model.eval()

def open_image(path, shape):
    image = Image.open(path).convert("RGB")
    return np.asarray(image.resize(shape))


In [12]:
def displayCircle(image, x, y, thickness=10, radius=25):
  if x == 0 and y == 0: return image
  cv2.circle(image, (x, y), radius, color, thickness)
  return image

cap = cv2.VideoCapture(os.path.join(project_path, 'data/DJI_0056_001.MP4'))
sidmoid = nn.Sigmoid()
count = -1
ball_list = []
all_ball_list = []
images = []
normal = (0,120,243)
count += 1
ret = True
while ret:
    count += 1
    ret, frame = cap.read()
    frame = cv2.GaussianBlur(frame,(3,3),0)
    kernel = np.ones((3, 3), np.uint8)
    frame = cv2.dilate(frame, kernel, iterations=1)
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (640, 360))
    images.append(image)
    if len(images) < 9:
        continue
    if len(images) > 9:
        images = images[1:]
    image = np.dstack(images)
    image = image.transpose((2, 0, 1))
    image = image[np.newaxis, :]
    image = torch.from_numpy(image/255.0).float().to(cuda0)
    output_heatpmap = model(image)
    output_numpy = output_heatpmap.squeeze().cpu().detach().numpy()
    max_y, max_x = detect(output_numpy, (360,640))
    color = normal
    all_ball_list.append(np.max(output_numpy))
    write_image = displayCircle(cv2.cvtColor(images[4], cv2.COLOR_RGB2BGR), max_x, max_y, 3, 3 )
    cv2.imwrite(os.path.join(project_path, 'output/{:05d}.jpg'.format(count)), write_image)

    if count % 100 == 0:
        print(datetime.datetime.now(), count)

2026-01-02 10:25:35.885665 100
2026-01-02 10:25:49.588472 200
2026-01-02 10:26:03.489792 300
2026-01-02 10:26:17.311786 400
2026-01-02 10:26:31.202286 500
2026-01-02 10:26:45.087258 600
2026-01-02 10:26:58.916641 700
2026-01-02 10:27:12.858859 800
2026-01-02 10:27:26.762818 900
2026-01-02 10:27:40.491833 1000
2026-01-02 10:27:54.379494 1100
2026-01-02 10:28:08.255595 1200
2026-01-02 10:28:22.122341 1300
2026-01-02 10:28:36.035385 1400
2026-01-02 10:28:49.953762 1500
2026-01-02 10:29:03.838713 1600
2026-01-02 10:29:17.947629 1700
2026-01-02 10:29:31.659963 1800
2026-01-02 10:29:45.592542 1900
2026-01-02 10:29:59.547316 2000
2026-01-02 10:30:13.458868 2100
2026-01-02 10:30:27.403892 2200
2026-01-02 10:30:41.213741 2300
2026-01-02 10:30:55.195034 2400
2026-01-02 10:31:08.985509 2500
2026-01-02 10:31:22.918190 2600
2026-01-02 10:31:36.820213 2700
2026-01-02 10:31:50.710908 2800
2026-01-02 10:32:04.517300 2900
2026-01-02 10:32:18.372768 3000
2026-01-02 10:32:32.148825 3100
2026-01-02 10:32:

error: OpenCV(4.12.0) /io/opencv/modules/imgproc/src/smooth.dispatch.cpp:618: error: (-215:Assertion failed) !_src.empty() in function 'GaussianBlur'


[0.012931065,
 0.03762221,
 0.05221333,
 0.056793172,
 0.029051999,
 0.0044612815,
 0.17692907,
 0.47441575,
 0.009763391,
 0.24719906,
 0.4976949,
 0.73659825,
 0.7686208,
 0.8364825,
 0.7739715,
 0.79097545,
 0.8796067,
 0.9037868,
 0.90465814,
 0.89300084,
 0.9116414,
 0.93933964,
 0.9415224,
 0.9152627,
 0.883796,
 0.9408579,
 0.94646305,
 0.9331262,
 0.9451959,
 0.9454783,
 0.9386558,
 0.9202186,
 0.9521803,
 0.933426,
 0.9430994,
 0.852228,
 0.85382766,
 0.6529461,
 0.719347,
 0.5002656,
 0.22618476,
 0.48039106,
 0.08392894,
 0.051841073,
 0.024340238,
 0.08142542,
 0.5022446,
 0.79896605,
 0.8868922,
 0.90782666,
 0.8958002,
 0.937416,
 0.87534875,
 0.8036413,
 0.9017259,
 0.9480546,
 0.4530836,
 0.38777122,
 0.61728936,
 0.8802497,
 0.8963008,
 0.9674068,
 0.87096274,
 0.94334674,
 0.9677213,
 0.9639747,
 0.9543004,
 0.9383467,
 0.9468607,
 0.8942246,
 0.9254462,
 0.9379916,
 0.9584225,
 0.9442579,
 0.9666284,
 0.9700019,
 0.9837964,
 0.9719953,
 0.97349507,
 0.97630095,
 0.95

In [ ]:
print(all_ball_list)
cap = cv2.VideoCapture('../data/PXL_20210313_065452204.mp4')
count = -1
count += 1
ret = True
while ret:
    count += 1
    ret, frame = cap.read()
    if count < 9:
        continue
    if sum(all_ball_list[count-60:count+60]) > 0.55 * 120:
        cv2.imwrite('output/{:05d}.jpg'.format(count), frame)

    if count % 100 == 0:
        print(datetime.datetime.now(), count)


[0.69693387, 0.5125023, 0.51443946, 0.60991347, 0.7183196, 0.7111761, 0.71085036, 0.71655244, 0.71997947, 0.7139537, 0.7137901, 0.71870893, 0.66917104, 0.7141232, 0.71806395, 0.71019447, 0.70060986, 0.7138033, 0.69653827, 0.5895845, 0.54876274, 0.508292, 0.5732209, 0.57108945, 0.52564114, 0.5514072, 0.62240374, 0.6028553, 0.5055754, 0.5454326, 0.61300886, 0.699193, 0.7113492, 0.6931392, 0.70030934, 0.7025916, 0.7089163, 0.71299285, 0.7130313, 0.7181724, 0.7143036, 0.7029558, 0.70805466, 0.7116239, 0.71552414, 0.66276515, 0.62492603, 0.66396433, 0.6856907, 0.6971344, 0.62780344, 0.6438666, 0.5207322, 0.53166634, 0.55531734, 0.6328626, 0.64907813, 0.5373614, 0.512509, 0.65130574, 0.7000864, 0.52019286, 0.5914964, 0.69548905, 0.6875865, 0.69700265, 0.7040666, 0.70394224, 0.7026153, 0.6860927, 0.69338584, 0.70758986, 0.6716041, 0.63623047, 0.6937465, 0.70909584, 0.7114596, 0.71385247, 0.7073732, 0.70259756, 0.51715964, 0.6827958, 0.67457265, 0.5512055, 0.52733177, 0.507542, 0.5045377, 0.53

In [ ]:
bound_mat = []
xtmp, ytmp = [], []
tmp = (0, 0, 0)
judge = False
for idx, (x, y, p) in enumerate(bound):
    if p > 0.7:
        tmp = compare(tmp, (x, y, p))
#         xtmp.append(x)
#         ytmp.append(y)
        judge = True
    elif judge:
        judge = False
        bound_mat.append(tmp)
        tmp = (0, 0, 0)
#         bound_mat.append([np.mean(xtmp), np.mean(ytmp), p])
#         xtmp, ytmp = [], []
    savefig(bound_mat, idx)

In [ ]:
def savefig(bound_mat, idx):
    if bound_mat:
        bound_array = np.array([[x[0] * 3, x[1] * 3] for x in bound_mat], dtype=np.float32)
        h, w, c = 1080, 1920, 3
        # test_3のテーブル
        bound_array = np.array([bound_array])
        src_pts = np.array([[500, 625], [300, 760] ,[1502, 760],[1295, 626]], dtype=np.float32)
        dst_pts = np.array([[0, 0], [0, h], [w, h], [w, 0]], dtype=np.float32)
        mat = cv2.getPerspectiveTransform(src_pts, dst_pts)
        mat = cv2.perspectiveTransform(bound_array, mat)
        plt.xlim([0,1920])
        plt.ylim([-1080, 0])
        plt.scatter(mat[0,:,0], -mat[0,:,1])
    plt.savefig('output2/{:05d}.png'.format(idx))

def compare(x, y):
    if x[2] > y[2]:
        return x
    else:
        return y

In [ ]:
plt.xlim([0,2000])
plt.ylim([-1100, 0])
plt.scatter(mat[0,:,0], -mat[0,:,1])

In [ ]:
images = sorted(glob.glob('output/*.jpg'))
balls = sorted(glob.glob('output2/*.png'))
for idx, (image, ball) in enumerate(zip(images, balls)):
    img = cv2.imread(image)
    b = cv2.imread(ball)
    b = cv2.resize(b, (280,140))
    img[0:b.shape[0],0:b.shape[1]] = b
    cv2.imwrite('output2/{:05d}.jpg'.format(idx), img)

# Task
Add code to cell `RpK0bhLeBoHq` to mount Google Drive and append the necessary paths to `sys.path` to resolve the `ModuleNotFoundError` for 'models' and 'utils'.

## mount_google_drive_and_update_path

### Subtask:
Mount Google Drive and add project paths to sys.path.


## Summary:

### Data Analysis Key Findings
*   Google Drive was successfully mounted, providing access to necessary files.
*   The `sys.path` was updated to include project-specific directories, resolving potential `ModuleNotFoundError` issues for 'models' and 'utils' modules.

### Insights or Next Steps
*   The environment is now correctly configured, allowing for the import and use of custom modules like 'models' and 'utils' in subsequent analysis steps.
